In [28]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment

### Load data with pandas

In [2]:
trm_ktrials = pd.read_excel("recreated_data.xlsx", sheet_name="ktrials", nrows=25)
trm_ftrials = pd.read_excel("recreated_data.xlsx", sheet_name="ftrials", nrows=25)
# trm_ctrials = pd.read_excel("recreated_data.xlsx", sheet_name="ctrials", nrows=25)

psychopy_data = pd.read_excel("PsychoPy_data.xlsx", sheet_name="PsychoPy_R", nrows=25)

### Load PsychoPy data with openpyxl to be able to save it without losing formatting

In [3]:
wb = load_workbook("PsychoPy_data.xlsx")
ws = wb["PsychoPy_R"]

In [4]:
# This is only use with the recreated data
trm_ktrials.rename(columns={'k_resp.ke…': 'k_resp.keys_raw'}, inplace=True)
trm_ftrials.rename(columns={'f_slider.re…': 'f_slider.response_mean'}, inplace=True)

In [5]:
COLS_FOR_KTRIALS = ['order', 'k_resp.keys_raw']
COLS_FOR_FTRIALS = ['order', 'f_slider.response_mean', 'f_slider.response_raw']
COLS_FOR_CTRIALS = ['order', 'c_slider.response_mean', 'c_slider.response_raw']
COLS_FOR_PSYCHOPY = {'ktrials': ['kOrder', 'kResp'],
                     'ftrials': ['fOrder', 'fSlider'],
                     'ctrials': ['cOrder', 'cSlider']}


### Getting only the file name to be use as a pk

In [6]:
trm_ktrials['img_file']  = trm_ktrials['img_file'].str[-7:]
trm_ftrials['img_file']  = trm_ftrials['img_file'].str[-7:]
# trm_ctrials['img_file']  = trm_ctrials['img_file'].str[-7:]

In [7]:
# Testing if the keys are unique
trm_ktrials[['img_file', 'Race', 'Gender'] + COLS_FOR_KTRIALS].head(5)

,img_file,Race,Gender,order,k_resp.keys_raw
0,BM1.png,Black,Man,21,d
1,BM2.png,Black,Man,10,d
2,BW1.png,Black,Woman,0,d
3,BW2.png,Black,Woman,12,d
4,LM1.png,Latino,Man,9,d


In [8]:
# Checking the destination sheet for the new columns
psychopy_data.head()

,StudyID,img_file,Race,Gender,Option,kOrder,kResp,fOrder,fSlider,cOrder,cSlider,fZ,cZ
0,9994,BM1.png,Black,Man,1,NaN,NaN,4,1.613333,11,2.180000,1.058364,1.092048
1,9994,BM2.png,Black,Man,2,NaN,NaN,20,1.686667,7,1.346667,1.135614,-0.143300
2,9994,BW1.png,Black,Woman,1,NaN,NaN,18,1.433333,17,0.746667,0.868754,-1.032751
3,9994,BW2.png,Black,Woman,2,NaN,NaN,22,0.980000,19,1.353333,0.391218,-0.133418
4,9994,LM1.png,Latino,Man,1,NaN,NaN,1,0.466667,0,1.993333,-0.149523,0.815330


In [9]:
keys = ['img_file',	'Race',	'Gender']

### Making the lookup series for the new columns to be added to the psychopy_data dataframe

In [10]:
ktrials_lookup_series = trm_ktrials.set_index(keys)['order']
ktrials_lookup_series_resp = trm_ktrials.set_index(keys)['k_resp.keys_raw']

In [11]:
ftrials_lookup_series = trm_ftrials.set_index(keys)['order']
ftrials_lookup_series_slider = trm_ftrials.set_index(keys)['f_slider.response_mean']

In [12]:
# ctrials_lookup_series = trm_ctrials.set_index(keys)['order']
# ctrials_lookup_series_slider = trm_ctrials.set_index(keys)['c_slider.response_mean']

In [13]:
psychopy_data['kOrder'] = psychopy_data.set_index(keys).index.map(ktrials_lookup_series)
psychopy_data['kResp'] = psychopy_data.set_index(keys).index.map(ktrials_lookup_series_resp)

In [14]:
psychopy_data['fOrder'] = psychopy_data.set_index(keys).index.map(ftrials_lookup_series)
psychopy_data['fSlider'] = psychopy_data.set_index(keys).index.map(ftrials_lookup_series_slider)

In [15]:
# psychopy_data['cOrder'] = psychopy_data.set_index(keys).index.map(ctrials_lookup_series)
# psychopy_data['cSlider'] = psychopy_data.set_index(keys).index.map(ctrials_lookup_series_slider)

In [17]:
psychopy_data.head()

,StudyID,img_file,Race,Gender,Option,kOrder,kResp,fOrder,fSlider,cOrder,cSlider,fZ,cZ
0,9994,BM1.png,Black,Man,1,21,d,4,1.613333,11,2.180000,1.058364,1.092048
1,9994,BM2.png,Black,Man,2,10,d,20,1.686667,7,1.346667,1.135614,-0.143300
2,9994,BW1.png,Black,Woman,1,0,d,18,1.433333,17,0.746667,0.868754,-1.032751
3,9994,BW2.png,Black,Woman,2,12,d,22,0.980000,19,1.353333,0.391218,-0.133418
4,9994,LM1.png,Latino,Man,1,9,d,1,0.466667,0,1.993333,-0.149523,0.815330


In [24]:
for row, value in enumerate(psychopy_data['kOrder'].iloc[:24], start=2):
    if pd.notna(value):
        ws.cell(row=row, column=psychopy_data.columns.get_loc('kOrder') + 1, value=value)

for row, value in enumerate(psychopy_data['kResp'].iloc[:24], start=2):
    if pd.notna(value):
        cell = ws.cell(row=row, column=psychopy_data.columns.get_loc('kResp') + 1, value=value)
        cell.alignment = Alignment(horizontal="right")

In [26]:
for row, value in enumerate(psychopy_data['fOrder'].iloc[:24], start=2):
    if pd.notna(value):
        ws.cell(row=row, column=psychopy_data.columns.get_loc('fOrder') + 1, value=value)

for row, value in enumerate(psychopy_data['fSlider'].iloc[:24], start=2):
    if pd.notna(value):
        ws.cell(row=row, column=psychopy_data.columns.get_loc('fSlider') + 1, value=value)

In [ ]:
# for row, value in enumerate(psychopy_data['cOrder'].iloc[:24], start=2):
#     if pd.notna(value):
#         ws.cell(row=row, column=psychopy_data.columns.get_loc('cOrder') + 1, value=value)
#
# for row, value in enumerate(psychopy_data['cSlider'].iloc[:24], start=2):
#     if pd.notna(value):
#         ws.cell(row=row, column=psychopy_data.columns.get_loc('cSlider') + 1, value=value)

In [27]:
wb.save("PsychoPy_data - Copy.xlsx")